<a href="https://colab.research.google.com/github/ubaid8878/Flyrank-ML-Internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ubaid8878/Flyrank-ML-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [ ]:
# Refresh / Content Opportunity Scoring

## Research Question

Can observable content and search signals help prioritize pages for refresh and CTR review?

The decision supported by this analysis is which pages should receive human review first.

This project focuses on decision-support rather than automated content changes. It does not attempt to prove Google's ranking algorithm or claim that refreshing a page will cause better rankings or traffic.

In [ ]:
## Abstract

This study asks whether observable content and search signals can help prioritize pages for refresh and CTR review. I use content age, days since last update, impressions, average position, CTR, and word count to rank pages for review. A simple Week-4 baseline is compared with a Decision Tree model, followed by a stricter client-grouped validation design. On the Week-5 random test split, the Decision Tree measured higher Precision@20 and Precision@50 than the baseline. The resulting recommendations are intended as human-reviewed decision-support rather than an automated content optimization system.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
## Data

This analysis uses the FlyRank ML Internship starter dataset.

The dataset contains approximately 30,000 content-page records and 44 columns.

The analysis uses observable signals that can be available before the modeled outcome:

- content_age_days
- days_since_last_update
- impressions_90d
- avg_position
- ctr
- word_count

The target is based on trend_direction, where a value of "down" is treated as the declining class.

Client identifiers are used only for grouped validation and are not used as predictive features.

Outcome-derived information such as trend_pct is excluded from model features because it would leak information about the target.

No client names, domains, private queries, credentials, or raw private exports are included.

In [4]:
import subprocess
import os

subprocess.run([
    "git", "clone",
    "https://github.com/ubaid8878/Flyrank-ML-Internship.git",
    "/content/Flyrank-ML-Internship"
], check=True)

os.chdir("/content/Flyrank-ML-Internship")

print("Current folder:", os.getcwd())
print("Dataset exists:", os.path.exists("data/raw/content_refresh_anonymized.csv"))

Current folder: /content/Flyrank-ML-Internship
Dataset exists: True


In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nDate columns available:")
print([c for c in df.columns if "date" in c.lower()])

print("\nTarget distribution:")
print(df["trend_direction"].value_counts())

Dataset shape: (30000, 44)
Rows: 30000
Columns: 44

Date columns available:
['days_since_last_update']

Target distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [ ]:
## Methodology

The target is a binary indicator:

- 1 = trend_direction is "down"
- 0 = otherwise

The Week-4 baseline uses two observable signals:

1. Staleness: days_since_last_update >= 180
2. Low CTR: CTR below the dataset median

Pages satisfying both conditions receive the highest baseline priority.

The Week-5 model is a Decision Tree using six observable features:

- content_age_days
- days_since_last_update
- impressions_90d
- avg_position
- ctr
- word_count

The original Week-5 experiment used an 80/20 stratified random split.

Week 6 added client-grouped validation so pages from the same client do not appear in both training and testing. This provides a more conservative test of generalization.

The primary ranking metrics are Precision@20 and Precision@50 because the practical goal is to prioritize a small number of pages for human review.

Leakage checks exclude trend_pct, trend_direction, client_id, and content_id from predictive features.

In [6]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
import numpy as np

# Target
y = df["trend_direction"].str.lower().eq("down").astype(int)

# Observable features only
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = df[features].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

# Client-grouped split
groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training clients:", groups.iloc[train_idx].nunique())
print("Testing clients:", groups.iloc[test_idx].nunique())

print(
    "Client overlap:",
    len(
        set(groups.iloc[train_idx])
        & set(groups.iloc[test_idx])
    )
)

Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Client overlap: 0


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
## Results

The Week-5 random test split produced the following measured results:

| Method | Precision@20 | Precision@50 |
|---|---:|---:|
| Week-4 Baseline | 0.50 | 0.44 |
| Decision Tree | 0.65 | 0.68 |

On that split, the Decision Tree measured higher precision than the baseline at both K values.

These results are measured on one evaluation split and should not be interpreted as a guarantee of future performance.

The client-grouped validation below provides a stricter assessment because clients are separated between training and testing.

In [7]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Train Decision Tree on grouped training data
tree = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

tree.fit(X_train, y_train)

# Model scores on grouped test set
tree_scores = tree.predict_proba(X_test)[:, 1]

# Week-4 baseline on the SAME grouped test set
test_df = df.iloc[test_idx].copy()

ctr_median = df["ctr"].median()

stale = test_df["days_since_last_update"] >= 180
low_ctr = test_df["ctr"] < ctr_median

baseline_scores = (
    stale.astype(int) * 2
    + low_ctr.astype(int)
)

print("GROUPED VALIDATION RESULTS")
print()

for k in [20, 50]:

    baseline_precision = precision_at_k(
        baseline_scores,
        y_test,
        k
    )

    tree_precision = precision_at_k(
        tree_scores,
        y_test,
        k
    )

    print(
        f"Precision@{k}: "
        f"Baseline={baseline_precision:.3f} | "
        f"Decision Tree={tree_precision:.3f}"
    )

GROUPED VALIDATION RESULTS

Precision@20: Baseline=0.500 | Decision Tree=0.600
Precision@50: Baseline=0.660 | Decision Tree=0.560


## 5. Limitations

*What this work cannot claim.*

In [ ]:
## Limitations

This analysis measures associations and ranking performance in the available dataset. It does not establish causation.

A stale page may have weak performance for reasons unrelated to freshness. Similarly, low CTR does not necessarily mean that content quality is poor.

The Decision Tree can combine several signals, but model performance may change across clients, topics, and time periods.

The client-grouped validation is more conservative than the original random split, but it is still one validation experiment.

The recommendations should therefore be interpreted as observed, measured, directional, and useful for decision-support.

The system should not automatically rewrite, delete, redirect, publish, or change pages. A human reviewer should make the final decision.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
## Ranked Recommendations

### 1. STALE_LOW_CTR — Review refresh and CTR

These pages are both stale and below the median CTR. They receive the highest priority in the baseline action score.

**Action:** REVIEW_REFRESH_AND_CTR

A human reviewer should check whether the content is still current and whether the search result presentation may deserve investigation.

---

### 2. STALE — Review refresh

These pages meet the staleness threshold but do not have below-median CTR.

**Action:** REVIEW_REFRESH

The reviewer should determine whether the information is outdated or whether there is a legitimate reason not to update it.

---

### 3. LOW_CTR — Review CTR

These pages have below-median CTR but are not classified as stale.

**Action:** REVIEW_CTR

The reviewer should investigate search intent, result presentation, and other contextual factors before deciding whether any change is appropriate.

---

### 4. NO_FLAG — No action

Pages without either baseline signal are not prioritized by this simple rule.

**Action:** NO_ACTION

These pages should not be automatically considered successful or unsuccessful.

In [ ]:
## Intended Use

The ranked queue is intended to help a content team decide which pages to inspect first.

It is not a production automation system.

### Human review is required before:

- refreshing content
- changing titles or metadata
- rewriting content
- merging pages
- pruning pages
- redirecting URLs

### What should NOT be automated

The model should not automatically publish content changes, delete pages, redirect pages, or claim that a change will improve Google rankings or traffic.

The model produces a prioritization signal, not a final business decision.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
import matplotlib.pyplot as plt

methods = [
    "Week-4 Baseline",
    "Decision Tree"
]

precision_20 = [
    0.50,
    0.65
]

plt.figure(figsize=(8, 5))
plt.bar(methods, precision_20)
plt.ylabel("Precision@20")
plt.title("Week-5 Model vs Week-4 Baseline")
plt.ylim(0, 1)
plt.show()

In [ ]:
## Reproducibility

The complete project is available in the public GitHub repository:

https://github.com/ubaid8878/Flyrank-ML-Internship

The repository contains the weekly notebooks used to develop the baseline, model, validation audit, and action playbook.

The capstone notebook brings these components together into the final analysis.

In [ ]:
## Acknowledgments & Data Credit

Built on the FlyRank ML Internship dataset.

Data and research context provided by FlyRank.

This project was completed as part of the FlyRank ML Internship.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
